# Super-resolved Sentinel-2 on demand

This notebook sends an area and a date to the SR service and gets back 2.5 m imagery made
from 10 m Sentinel-2. It takes about 10 minutes, most of which is waiting for the GPUs.

1. Install the client and connect with your email address.
2. Estimate the cost of an area and see which Sentinel-2 scenes are available.
3. Run stage 1, which is fast and multi-image, then add stage 2, the diffusion refinement.
4. Look at the result, open it as an xarray, and download the files.

In [ ]:
%pip install -q "sr-client[geo,viz] @ git+https://github.com/adb-sr/sr-client@v0.1.0"

## 1. Connect

Run the next cell and type your **email address**: the service gives you a key for it (the
same email always gets the same key). If you were given a key (`srk_...`), type that instead,
or store it in Colab under **Secrets** as `SR_API_KEY`.

In [ ]:
# @title Connect (run this cell, then type your email address) { display-mode: "form" }
from sr_client import SRClient, area_around

# ---- connecting to the SR service: your email is enough ---------------------------------------
# (tools/connect.py, pasted into every notebook that uses the service. The sign-up is the
# workshop's: the service gives each email a key, and the same email gets the same key back.)
import os

SERVICE_URL = "https://sr-service-dev--sr-service-api.modal.run"


def sr_signup(email, url=None):
    """Get the workshop key of an email address from the service and keep it, with the
    service address, in the environment (SR_API_KEY, SR_API_URL), where the client finds
    them. Raises RuntimeError with the service's message if it does not work."""
    import json
    import urllib.error
    import urllib.request

    url = (url or os.environ.get("SR_API_URL") or SERVICE_URL).rstrip("/")
    req = urllib.request.Request(f"{url}/v1/signup",
                                 data=json.dumps({"email": email.strip()}).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            got = json.load(r)
    except urllib.error.HTTPError as exc:
        try:
            detail = json.load(exc)["detail"]
        except Exception:
            detail = f"the service answered with error {exc.code}"
        raise RuntimeError(detail if isinstance(detail, str)
                           else "that does not look like an email address") from None
    except OSError as exc:
        raise RuntimeError(f"the service could not be reached ({exc})") from None
    os.environ["SR_API_KEY"], os.environ["SR_API_URL"] = got["key"], url
    return got


def _stored_key():
    """A key that is already there: in the environment, or in Colab's Secrets (SR_API_KEY)."""
    if os.environ.get("SR_API_KEY"):
        return os.environ["SR_API_KEY"]
    try:
        from google.colab import userdata
        return userdata.get("SR_API_KEY") or ""
    except Exception:
        return ""


def connect():
    """Ask for your email address, get your key from the service, and return the connected
    client. (If you were given a key, type it instead of the email.)"""
    from sr_client import SRClient

    os.environ.setdefault("SR_API_URL", SERVICE_URL)
    key = _stored_key()
    while not key:
        answer = input("Your email address: ").strip()
        if answer.startswith("srk_"):
            key = answer
            break
        try:
            key = sr_signup(answer)["key"]
        except RuntimeError as exc:
            print(f"That did not work: {exc}. Try again, or ask an instructor.")
    os.environ["SR_API_KEY"] = key
    sr = SRClient()
    me = sr.me()
    print(f"Connected as {me['owner']}: areas up to {me['limits']['max_area_km']} km per side, "
          f"${me['budget']['remaining_usd']:.2f} of budget left.")
    return sr


sr = connect()
sr.me()

## 2. Pick an area and estimate

`area_around(lat, lon, km)` makes a square. You can also pass a bounding box
`[W, S, E, N]` or any GeoJSON polygon. Estimates are free: they show the tile count, the cost
(assuming nothing is cached yet), the Sentinel-2 scenes in the time window, and warnings that
are likely to fire.

In [ ]:
area = area_around(14.5995, 120.9842, km=2)   # Manila; try your own place
target_date = "2026-04-15"            # dry season: many clear scenes

est = sr.estimate(area, target_date)
print(f"{est['tiles']} tiles, about ${est['estimated_usd']:.2f}")
for s in est["scenes"]:
    print(f"  {s['date']}  cloud {s['cloud_cover']:5.1f} %  {s['id']}")
for w in est["warnings_likely"]:
    print("warning:", w["message"])
for p in est["problems"]:
    print("cannot submit:", p)

## 3. Run it

Stage 1 alone takes a minute or two and costs very little. `job.upgrade()` then adds stage 2,
the diffusion refiner, which reuses stage 1's tiles. You can also submit both stages at once,
which is the default.

In [ ]:
job = sr.submit(area, target_date, stages=["1"])
job.wait()
job.show()

In [ ]:
job.upgrade().wait()
job.show()

## 4. Use the result

`job.open()` reads a GeoTIFF straight from its download link as an xarray `DataArray` in
reflectance units. Below, the 10 m input and both stages are compared over a zoomed window.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

lr = job.open("lr_reference.tif")
s1 = job.open("stage1_sr.tif")
s2 = job.open("stage2_sr.tif")

def rgb(da, y0, x0, size):
    a = da.isel(band=[0, 1, 2], y=slice(y0, y0 + size), x=slice(x0, x0 + size)).values
    return np.clip(np.nan_to_num(np.moveaxis(a, 0, -1)) / 0.3, 0, 1)

h, w = s2.sizes["y"], s2.sizes["x"]
y0, x0, size = h // 2 - 100, w // 2 - 100, 200          # a 500 m window at 2.5 m
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, img) in zip(axes, [("Sentinel-2, 10 m", rgb(lr, y0 // 4, x0 // 4, size // 4)),
                                   ("stage 1, 2.5 m", rgb(s1, y0, x0, size)),
                                   ("stage 2, 2.5 m", rgb(s2, y0, x0, size))]):
    ax.imshow(img, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.show()

red, nir = s2.sel(band=1), s2.sel(band=4)
ndvi = ((nir - red) / (nir + red)).rename("NDVI")
ndvi.attrs = {}
fig, ax = plt.subplots(figsize=(7, 6))
ndvi.plot(ax=ax, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax.set_title("NDVI from the stage-2 output")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

In [ ]:
for w in job.warnings:
    print(f"[{w['code']}] {w['message']}")

paths = job.download("sr_results")
paths

In Colab, the files are in the file browser on the left, under `sr_results/<job id>/`.
`manifest.json` lists the scenes used for every tile, the model versions and the cost.
`sr.me()` shows how much budget your key has left, and `sr.jobs()` lists your jobs.

Results contain modified Copernicus Sentinel data.

In [ ]:
sr.me()["budget"]